------------------------------------------Nadav 24/1/26---------------------------------------------------------------

In [ ]:
import pandas as pd

In [ ]:
df_work = pd.read_csv(r"C:\Users\nadid\Downloads\df_raw_with_start_periods_19_1.csv")

df_work.shape

(3617891, 36)

In [ ]:
df_work[["event_type", "event_type_desc"]].head(5)

,event_type,event_type_desc
0,1,visitor_line
1,2,rep_line
2,1,visitor_line
3,2,rep_line
4,1,visitor_line


In [ ]:
customers = df_work[df_work["event_type"] == 1].reset_index(drop = True)
agents = df_work[df_work["event_type"] == 2].reset_index(drop = True)
print(customers.shape, agents.shape)

(1941445, 36) (1676446, 36)


GPT SUGGESTION - GROUP BY AND AGG Function, instead of using apply - USE THIS! MUCH FASTER!

In [ ]:
group_id = (
    (customers[column_to_sum_by] != customers[column_to_sum_by].shift()) |
    (customers['id_session'] != customers['id_session'].shift())
).cumsum()


In [ ]:
agg_dict = {}

# sum columns
for col in columns_to_sum:
    agg_dict[col] = 'sum'

# keep first value for all other columns
for col in columns_to_keep_original_val:
    agg_dict[col] = 'first'

# count rows per group
agg_dict['n_messages'] = ('id_session', 'size')


In [ ]:
start_time = time.time()

merged_df_gpt = (
    customers
    .groupby(group_id, sort=False)
    .agg(
        **{col: (col, 'sum') for col in columns_to_sum},
        **{col: (col, 'first') for col in columns_to_keep_original_val},
        n_messages=('id_session', 'size')
    )
    .reset_index(drop=True)
)


end_time = time.time()
print(f"Execution time: {end_time - start_time:.2f} seconds")


Execution time: 14.24 seconds


In [ ]:
(merged_df_gpt.shape[0] / customers.shape[0])

0.877584479601534

 #### Random Checking:

In [ ]:
merged_df_gpt[["id_session", "end_date" , "n_messages", "start_period", "sentiment", "number_words", "duration"]].sort_values(by = "n_messages", ascending = False).head(15)

,id_session,end_date,n_messages,start_period,sentiment,number_words,duration
594381,100063760,27/05/2017 14:20:41,19,27/05/2017 14:20:06,14,781,153
510033,100054738,26/05/2017 03:11:45,18,26/05/2017 03:11:44,-30,285,432
246451,100026351,22/05/2017 16:50:42,17,22/05/2017 16:45:42,-7,426,655
194301,100020761,31/05/2017 12:25:22,17,31/05/2017 12:25:22,-11,356,297
999464,100113656,31/05/2017 02:37:36,16,31/05/2017 02:37:23,-4,356,504
96348,100010358,29/05/2017 16:08:13,15,29/05/2017 16:05:52,4,2702,621
1431389,100190857,14/05/2017 15:22:27,15,14/05/2017 15:22:26,-33,323,1287
753801,100081076,13/05/2017 16:04:54,13,13/05/2017 16:04:06,-6,282,233
1505113,100214967,06/05/2017 21:12:41,13,06/05/2017 19:21:29,-10,198,24278
302243,100032322,26/05/2017 18:18:42,13,26/05/2017 18:16:06,-13,458,687


In [ ]:
cols = ["id_session", "end_date", "event_type_desc" ,"start_period", "duration", "sentiment", "number_words", "event_id"]

df_work[df_work["id_session"] == 100010358][cols]

,id_session,end_date,event_type_desc,start_period,duration,sentiment,number_words
206543,100010358,29/05/2017 16:00:32,visitor_line,29/05/2017 16:00:32,0,0,14
206544,100010358,29/05/2017 16:00:35,visitor_line,29/05/2017 16:00:32,3,-1,246
206545,100010358,29/05/2017 16:00:56,visitor_line,29/05/2017 16:00:32,21,0,77
206546,100010358,29/05/2017 16:01:55,visitor_line,29/05/2017 16:00:32,59,1,87
206547,100010358,29/05/2017 16:05:46,visitor_line,29/05/2017 16:00:32,231,0,55
206548,100010358,29/05/2017 16:05:52,rep_line,29/05/2017 16:00:32,14,0,56
206549,100010358,29/05/2017 16:08:13,visitor_line,29/05/2017 16:05:52,51,0,349
206550,100010358,29/05/2017 16:08:30,visitor_line,29/05/2017 16:05:52,17,-1,40
206551,100010358,29/05/2017 16:11:03,visitor_line,29/05/2017 16:05:52,103,0,395
206552,100010358,29/05/2017 16:11:07,visitor_line,29/05/2017 16:05:52,4,1,32


In [ ]:
# df_final = pd.concat([merged_df_gpt, agents])

df_final = df_final.reset_index(drop = True)

df_final.shape

(3380228, 37)

In [ ]:
df_final["n_messages"].isna().sum()

1676446

In [ ]:
final_columns_order = [ 'id_site', 'id_session', 'id_visitor', "id_rep" ,
                       'start_date', 'start_time', 'end_date', 'end_time',
                       'event_type', 'event_type_desc', 'sentiment', 'duration', 
                       'number_words',  'start_period', 'n_messages', 'number_chars', 
                       'number_lines', 'answer_canned', 'accept_date', 'accept_time', 
                       'read_date', 'read_time',  'outcome', 'outcome_desc', 
                       'subsession', 'delay', 'id_rep_code', 'sentiment_type', 
                       'id_agent', 'id_agent_code', 'source_file', 'queue_exit_date', 
                       'assignment_date', 'event_id', 'first_msg_dummy', 'old_id_rep', 
                       'chat_start_date']

In [ ]:
# df_final = df_final[final_columns_order]
cols = ["id_session", "end_date"]
df_final.sort_values(by = cols)
df_final.head(5)

,id_site,id_session,id_visitor,id_rep,start_date,start_time,end_date,end_time,event_type,event_type_desc,sentiment,duration,number_words,start_period,n_messages,number_chars,number_lines,answer_canned,accept_date,accept_time,read_date,read_time,outcome,outcome_desc,subsession,delay,id_rep_code,sentiment_type,id_agent,id_agent_code,source_file,queue_exit_date,assignment_date,event_id,first_msg_dummy,old_id_rep,chat_start_date
0,1,100000002,200064419,30001090,24/05/2017 22:57:02,1495666622,24/05/2017 22:57:02,1495666622,1,visitor_line,0,0,2,24/05/2017 22:57:02,1.0,7,0,3,01/01/1970,0,01/01/1970,0,1,served,1,0,2,3,30001090,1091,D24052017,NaN,24/05/2017 22:57:04,0,1,NaN,24/05/2017 22:57:02
1,1,100000002,200064419,30001090,24/05/2017 22:57:53,1495666673,24/05/2017 22:58:30,1495666710,1,visitor_line,0,37,15,24/05/2017 22:57:31,1.0,66,0,2,24/05/2017 22:57:31,1495666651,24/05/2017 22:57:31,1495666651,1,served,1,0,1091,3,30001090,1091,D24052017,NaN,None,5,0,NaN,24/05/2017 22:57:02
2,1,100000002,200064419,30001090,24/05/2017 23:00:21,1495666821,24/05/2017 23:00:26,1495666826,1,visitor_line,0,5,20,24/05/2017 22:59:26,1.0,81,0,2,24/05/2017 22:59:41,1495666781,24/05/2017 22:59:41,1495666781,1,served,1,0,1091,3,30001090,1091,D24052017,NaN,None,9,0,NaN,24/05/2017 22:57:02
3,1,100000002,200064419,30001090,24/05/2017 23:01:24,1495666884,24/05/2017 23:01:57,1495666917,1,visitor_line,0,33,4,24/05/2017 23:01:24,1.0,18,0,2,24/05/2017 23:01:34,1495666894,24/05/2017 23:01:34,1495666894,1,served,1,0,1091,3,30001090,1091,D24052017,NaN,None,12,0,NaN,24/05/2017 22:57:02
4,1,100000002,200064419,30001090,24/05/2017 23:02:33,1495666953,24/05/2017 23:02:49,1495666969,1,visitor_line,0,16,1,24/05/2017 23:02:33,1.0,3,0,2,24/05/2017 23:02:40,1495666960,24/05/2017 23:02:40,1495666960,1,served,1,0,1091,3,30001090,1091,D24052017,NaN,None,15,0,NaN,24/05/2017 22:57:02


In [ ]:
df_final = df_final.reset_index(drop = True)

In [ ]:
df_final.to_csv(r"C:\Users\nadid\OneDrive - Technion\Desktop\Nadav\Studies\Technion\service_systems\project\final_files\after_unification_of_messages_nadav_24_01.csv", index = False)